# Chapter 13 — Problem Set 1: Solutions

Reference implementations and short explanations for each problem in `problem_set_1.ipynb`.

---
*Generated by Berta AI*

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

import numpy as np
import pandas as pd

CORPUS_PATH = os.path.join('..', '..', 'datasets', 'sample_corpus.txt')
with open(CORPUS_PATH) as f:
    raw = f.read()
pattern = re.compile(r'^\[(doc-\d+)\]\s*(.+)$', re.MULTILINE | re.DOTALL)
documents = {}
for para in re.split(r'\n\s*\n', raw):
    m = pattern.match(para.strip())
    if m:
        documents[m.group(1)] = m.group(2).strip()
print('Corpus:', len(documents), 'docs')

## Problem 1 — Cosine similarity from scratch

The trick is to short-circuit when either norm is zero, which avoids divide-by-zero.

In [ ]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

assert abs(cosine_similarity([1, 0], [1, 0]) - 1.0) < 1e-6
assert abs(cosine_similarity([1, 0], [0, 1])) < 1e-6
assert abs(cosine_similarity([1, 0], [-1, 0]) + 1.0) < 1e-6
assert cosine_similarity([0, 0], [1, 1]) == 0.0
print('OK')

## Problem 2 — Fixed-size chunker

Step by `size - overlap` so the next chunk begins inside the previous one. Stop early when we run off the end.

In [ ]:
def chunk_text(text, size=200, overlap=40):
    if size <= 0 or overlap >= size:
        raise ValueError('size > overlap > 0 required')
    chunks = []
    step = size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + size]
        if not chunk.strip():
            continue
        chunks.append(chunk)
        if start + size >= len(text):
            break
    return chunks

sample = "abcdefghijklmnopqrstuvwxyz" * 4
print(len(chunk_text(sample, 20, 5)), 'chunks')
print(chunk_text(sample, 20, 5)[:2])

## Problem 3 — Encode and retrieve

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cos

ids = list(documents.keys())
texts = list(documents.values())

vec = TfidfVectorizer(stop_words='english').fit(texts)
M = vec.transform(texts)

def top_k(query, k=3):
    q = vec.transform([query])
    sims = sk_cos(q, M).ravel()
    order = np.argsort(-sims)[:k]
    return [ids[i] for i in order]

print(top_k('What is HyDE?', k=3))

## Problem 4 — hit@k

In [ ]:
queries_df = pd.read_csv(os.path.join('..', '..', 'datasets', 'queries.csv'))
queries_df['relevant'] = queries_df['relevant_doc_ids'].str.split('|')

def hit_at_k(retrieved, gold, k):
    return int(any(r in set(gold) for r in retrieved[:k]))

scores = {1: 0, 3: 0, 5: 0}
for _, row in queries_df.iterrows():
    rids = top_k(row['query'], k=5)
    for k in scores:
        scores[k] += hit_at_k(rids, row['relevant'], k)

n = len(queries_df)
for k, v in scores.items():
    print(f'hit@{k} = {v}/{n} = {v/n:.2f}')

## Problem 5 — Chunk size effect

In [ ]:
def evaluate_chunk_size(chunk_size):
    chunked_texts, chunked_doc_ids = [], []
    for did, t in documents.items():
        for c in chunk_text(t, size=chunk_size, overlap=chunk_size // 5):
            chunked_texts.append(c)
            chunked_doc_ids.append(did)

    v = TfidfVectorizer(stop_words='english').fit(chunked_texts)
    M = v.transform(chunked_texts)

    hit3 = 0
    for _, row in queries_df.iterrows():
        sims = sk_cos(v.transform([row['query']]), M).ravel()
        # Aggregate chunk scores up to doc level by max
        order = np.argsort(-sims)
        seen = []
        for i in order:
            if chunked_doc_ids[i] not in seen:
                seen.append(chunked_doc_ids[i])
            if len(seen) == 3:
                break
        if any(r in row['relevant'] for r in seen):
            hit3 += 1
    return hit3 / len(queries_df)

sizes = [100, 200, 400]
results = {s: evaluate_chunk_size(s) for s in sizes}
print(results)

# Smaller chunks usually help precision but hurt recall when an answer
# spans sentence boundaries. The sweet spot for this corpus is around 200.


## Problem 6 — Citation prompt template

In [ ]:
def build_prompt(question, contexts):
    ctx_block = "\n".join(f"[{cid}] {text}" for cid, text in contexts) or "(no context)"
    return (
        "You are a precise assistant. Use ONLY the context below to answer.\n"
        "Cite the chunk identifier in square brackets after each claim.\n"
        "If the context does not contain the answer, reply:\n"
        "  \"I don't know based on the provided context.\"\n\n"
        f"Context:\n{ctx_block}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

print(build_prompt(
    'What is HyDE?',
    [('doc-011', documents['doc-011'])]
))

---
*Generated by Berta AI*